# Vertical integration: RNA + ADT

Vertical integration combines modalities measured in the same cells. This tutorial is for RNA and surface protein, as in CITE-seq. It runs Matilda and sciPENN on `D11`, a CITE-seq dataset of 2,864 cells, and then on data in your own format.

Methods run on Linux. On macOS or Windows, [open this notebook in Colab](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_vertical_rna_adt.ipynb).

## 1. Install

This cell installs `multibench-sc`. It keeps the numpy and pandas that are already installed, so Colab needs no restart.

<details>
<summary>Details</summary>

On Colab, choose a GPU runtime before you run the notebook: Runtime -> Change runtime type -> T4 GPU. On a CPU runtime, an environment that has a smaller CPU build gets that build, and training methods run slower.

</details>

In [ ]:
import numpy, pandas
%pip install -q multibench-sc numpy=={numpy.__version__} pandas=={pandas.__version__}

## 2. Download the data and the environments

`mtb.data.fetch` downloads `D11` (11 MB) once. Each method runs in its own environment. `mtb.env.install` downloads the environments for Matilda and sciPENN, with no conda needed. The download is 1.1 GB on a computer without a GPU and 5.1 GB on one with a GPU.

<details>
<summary>Details</summary>

`state` is `PACKED` for an environment downloaded now and `have` for one that was already there. Without `dry_run=False`, `mtb.env.install` downloads nothing and returns the plan with its sizes.

Environments go to `mtb.config.DEFAULT.envs_dir`. To use another disk, set it before this cell.

From a terminal, `multibench env install --methods Matilda,sciPENN --packed --run` does the same.

</details>

In [ ]:
import pandas as pd
import multibench as mtb

mtb.data.fetch("D11")

In [ ]:
METHODS = ["Matilda", "sciPENN"]
envs = mtb.env.install(METHODS, dry_run=False)
pd.DataFrame(envs)[["env", "methods", "state"]]

## 3. Run the methods

`run_all` runs each method on `D11` and scores its output with the scIB metrics. `res.summary` has one row per method with its status, run time and scores. Higher is better for every metric.

<details>
<summary>Details</summary>

Each method writes an embedding: a table of numbers with one row per cell. `run_all` scores it against the cell-type labels of `D11`.

The clustering metrics `ARI`, `NMI`, `ASW`, `iASW`, `iF1` and `cLISI` measure how well the embedding separates the cell types. The batch metrics `ASW_batch`, `GC` and `iLISI` measure how well the batches mix. They appear only when the data has several batches.

`res.failures` says why a method failed. `params={"Method": {"key": value}}` sets a method's parameters, and `mtb.params_for` lists them. Many methods take none.

`mtb.load_batch("out/D11")` reloads these results later without running anything.

</details>

In [ ]:
res = mtb.run_all("D11", "vertical", methods=METHODS,
                  out_dir="out/D11")
res.summary

## 4. Plot

`res.plot()` draws the scores as a bubble table. Circle size shows the rank within a column, and bigger is better. The fill compares a value with the other rows in the same column.

<details>
<summary>Details</summary>

The lightest fill is the lowest value in this figure, not zero.

Metrics are grouped by family: blue for dimension reduction and clustering, green for batch correction. Each family starts with an `Overall` bar. Its length and colour both show the family score.

A column whose rows all hold the same value is drawn grey, and the note under the figure names it. A figure of one method is grey everywhere.

</details>

In [ ]:
res.plot()

## 5. Your own data

Your data needs raw counts for both modalities and a cell type for each cell. `mtb.io.export_dataset` writes an AnnData as a dataset folder, and `run_all` runs on that folder.

<details>
<summary>Details: export</summary>

Keep several samples in one folder, without `batch=`. To score the batch mixing, pass the batch column to `run_all(batch=...)`.

The folder name, `MYCITE`, is the dataset name for `run_all`, and `data_path` is the folder that holds it.

`mtb.scan("MYCITE", "vertical", data_path="mydata")` checks the folder and the environments without running anything. Its `reason` column says what is missing.

`overwrite=True` replaces the files of an earlier run of this cell. Without it, `export_dataset` raises `FileExistsError` rather than replace a file.

</details>

Here an AnnData made from 60% of `D11`'s cells takes the place of your data:

In [ ]:
import scanpy as sc

d = mtb.config.DEFAULT.data_path / "D11"
adata = mtb.io.read_canonical(d / "rna.h5")
adata.obsm["protein"] = mtb.io.read_canonical(d / "adt.h5").to_df()
adata.obs["celltype"] = pd.read_csv(d / "cty.csv")["x"].values
adata = sc.pp.subsample(adata, fraction=0.6, random_state=0, copy=True)
adata

Write the folder, then run the same methods on it:

In [ ]:
mtb.io.export_dataset(adata, "mydata/MYCITE", rna="X", adt="obsm:protein",
                      labels="obs:celltype", overwrite=True)

In [ ]:
mine = mtb.run_all("MYCITE", "vertical", methods=METHODS, data_path="mydata",
                   out_dir="out/MYCITE")
mine.summary

In [ ]:
mine.plot()

## 6. Stored scores

The package ships stored scores for 13 methods on `D11`. `load_results` reads them and `mtb.plot.bubble` draws them, without running anything.

<details>
<summary>Details</summary>

`source="rerun"` reads the package's own runs of the methods. `source="published"`, the default, reads the published scIB tables, which hold 6 methods for `D11`.

The stored scores used the `leidenalg` backend for Leiden clustering, and `run_all` uses `igraph` by default. The two backends can move ARI by up to about 0.1. To compare your runs with these scores, set `mtb.config.DEFAULT.leiden_flavor = "leidenalg"` before `run_all`.

</details>

In [ ]:
long = mtb.load_results("vertical", dataset="D11", source="rerun")
mtb.plot.bubble(long)

## 7. Every method of this task

14 methods have a variant for vertical RNA + ADT: Concerto, MOFA2, Matilda, Multigrate, Seurat_WNN, UINMF, VIMCCA, moETM, scMDC, scMM, scMSI, scMoMaT, sciPENN and totalVI. Each runs on `D11`. To run them all, set `METHODS` to that list in section 2 and run the notebook again. That downloads 14 envs, 32.5 GB to download on a CPU host, 40.8 GB on a GPU host.

[Every method on `D11`](https://dsichang.github.io/scMultiBench/tutorials/vertical_rna_adt_all/) shows that run.

## Troubleshooting

`res.failures` lists each method that failed or was skipped. Its `error` column ends with the method's error output.

<details>
<summary>Details</summary>

| symptom | fix |
|---|---|
| `env.install` refuses on macOS or Windows | methods run only on Linux: use Colab or a Linux machine |
| a method needs an NVIDIA GPU | choose a GPU runtime on Colab, or a machine with a GPU |
| a method fails on a small dataset | `mtb.scan` names what the method needs from the data's size in its `caveat` column |
| a warning that values are not whole numbers | export raw counts, for example with `rna="layer:counts"` |
| `... matrix/data as cells x features` | the matrix is transposed: export it again with `mtb.io.export_dataset` |
| a method times out | raise `timeout=` in `run_all` |
| low `label_order_confidence` | several label files fit the cell count: check `label_order_candidates` in `res.results` |
| batch metrics use the wrong batches | `res.rescore(batch=my_vector)` scores again without running the methods |

</details>

## Next steps

- `mtb.cite(METHODS)` returns the citations for the benchmark and the methods you ran.
- The other tutorials: [vertical RNA + ATAC](https://dsichang.github.io/scMultiBench/tutorials/vertical_rna_atac/), [vertical RNA + ADT + ATAC](https://dsichang.github.io/scMultiBench/tutorials/vertical_rna_adt_atac/), [diagonal RNA + ATAC](https://dsichang.github.io/scMultiBench/tutorials/diagonal_rna_atac/), [mosaic RNA + ATAC](https://dsichang.github.io/scMultiBench/tutorials/mosaic_rna_atac/), [mosaic RNA + ADT + ATAC](https://dsichang.github.io/scMultiBench/tutorials/mosaic_rna_adt_atac/), [mosaic RNA + ADT](https://dsichang.github.io/scMultiBench/tutorials/mosaic_rna_adt/), [cross RNA + ADT](https://dsichang.github.io/scMultiBench/tutorials/cross_rna_adt/).
- The guides: [run](https://dsichang.github.io/scMultiBench/tutorials/run/), [evaluate](https://dsichang.github.io/scMultiBench/tutorials/evaluate/), [plot](https://dsichang.github.io/scMultiBench/tutorials/plot/) and [discover methods](https://dsichang.github.io/scMultiBench/tutorials/discover/).
- The [interactive explorer](https://shiny.maths.usyd.edu.au/scMultiBench/) has the full benchmark's rankings, with no install needed.